In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv(r'D:\Projects\Exoplanet-Detection\data\raw\cumulative.csv')

print(df.shape)
df.head()

(9564, 50)


,rowid,kepid,kepoi_name,kepler_name,koi_disposition,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,koi_fpflag_co,...,koi_steff_err2,koi_slogg,koi_slogg_err1,koi_slogg_err2,koi_srad,koi_srad_err1,koi_srad_err2,ra,dec,koi_kepmag
0,1,10797460,K00752.01,Kepler-227 b,CONFIRMED,CANDIDATE,1.000,0,0,0,...,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
1,2,10797460,K00752.02,Kepler-227 c,CONFIRMED,CANDIDATE,0.969,0,0,0,...,-81.0,4.467,0.064,-0.096,0.927,0.105,-0.061,291.93423,48.141651,15.347
2,3,10811496,K00753.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,...,-176.0,4.544,0.044,-0.176,0.868,0.233,-0.078,297.00482,48.134129,15.436
3,4,10848459,K00754.01,NaN,FALSE POSITIVE,FALSE POSITIVE,0.000,0,1,0,...,-174.0,4.564,0.053,-0.168,0.791,0.201,-0.067,285.53461,48.285210,15.597
4,5,10854555,K00755.01,Kepler-664 b,CONFIRMED,CANDIDATE,1.000,0,0,0,...,-211.0,4.438,0.070,-0.210,1.046,0.334,-0.133,288.75488,48.226200,15.509


In [3]:
df_binary = df[
    df['koi_disposition'].isin([
        'CONFIRMED',
        'FALSE POSITIVE'
    ])
].copy()

df_binary['label'] = (
    df_binary['koi_disposition'] == 'CONFIRMED'
).astype(int)

print(df_binary.shape)
print(df_binary['label'].value_counts())

(7316, 51)
label
0    5023
1    2293
Name: count, dtype: int64


In [4]:
drop_cols = [
    'rowid',
    'kepid',
    'kepoi_name',
    'kepler_name',
    'koi_disposition',
    'koi_pdisposition',
    'koi_score',
    'label'
]

drop_cols += [
    col for col in df_binary.columns
    if col.endswith(('err1', 'err2'))
]

feature_cols = [
    'koi_period',
    'koi_prad',
    'koi_depth',
    'koi_duration',
    'koi_impact',
    'koi_insol',
    'koi_teq',
    'koi_steff',
    'koi_srad',
    'koi_model_snr'
]

print("Selected Features:")

print(feature_cols)
print(f"Number of selected features: {len(feature_cols)}")
print("\nFirst few features:\n")
print(feature_cols[:15])

Selected Features:
['koi_period', 'koi_prad', 'koi_depth', 'koi_duration', 'koi_impact', 'koi_insol', 'koi_teq', 'koi_steff', 'koi_srad', 'koi_model_snr']
Number of selected features: 10

First few features:

['koi_period', 'koi_prad', 'koi_depth', 'koi_duration', 'koi_impact', 'koi_insol', 'koi_teq', 'koi_steff', 'koi_srad', 'koi_model_snr']


In [5]:
X = df_binary[feature_cols].copy()
y = df_binary['label'].copy()
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

Feature matrix shape: (7316, 10)
Target shape: (7316,)


In [6]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)
print("Missing values before:", X.isnull().sum().sum())
print("Missing values after:", X_imputed.isnull().sum().sum())

Missing values before: 2359
Missing values after: 0


In [7]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (4974, 10)
Validation shape: (878, 10)
Test shape: (1464, 10)


In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("Scaling complete!")
print("Train mean:", X_train_scaled.mean())
print("Train std:", X_train_scaled.std())

Scaling complete!
Train mean: 4.714095351846262e-17
Train std: 1.0


In [9]:
print("Before SMOTE:")
print(y_train.value_counts())
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_scaled,
    y_train
)
print("\nAfter SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

Before SMOTE:
label
0    3415
1    1559
Name: count, dtype: int64

After SMOTE:
label
0    3415
1    3415
Name: count, dtype: int64


In [10]:
X_train_df = pd.DataFrame(
    X_train_resampled,
    columns=feature_cols
)
X_val_df = pd.DataFrame(
    X_val_scaled,
    columns=feature_cols
)
X_test_df = pd.DataFrame(
    X_test_scaled,
    columns=feature_cols
)

X_train_df['label'] = y_train_resampled.values
X_val_df['label'] = y_val.values
X_test_df['label'] = y_test.values

X_train_df.to_csv(r"D:\Projects\Exoplanet-Detection\data\processed\train.csv",index=False)
X_val_df.to_csv(r"D:\Projects\Exoplanet-Detection\data\processed\val.csv",index=False)
X_test_df.to_csv(r"D:\Projects\Exoplanet-Detection\data\processed\test.csv",index=False)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


In [11]:
confirmed_examples = df_binary[
    df_binary['label'] == 1][
    [
        'koi_period',
        'koi_prad',
        'koi_depth',
        'koi_duration'
    ]
]

confirmed_examples.head(10)

,koi_period,koi_prad,koi_depth,koi_duration
0,9.488036,2.26,615.8,2.95750
1,54.418383,2.83,874.8,4.50700
4,2.525592,2.75,603.3,1.65450
5,11.094321,3.90,1517.5,4.59450
6,4.134435,2.77,686.0,3.14020
7,2.566589,1.59,226.5,2.42900
9,16.068647,5.76,4914.3,3.53470
10,2.470613,13.04,14231.0,1.74319
11,2.204735,16.10,6674.7,3.88864
12,3.522498,14.59,9145.7,3.19843


In [12]:
import joblib

joblib.dump(scaler, r"D:\Projects\Exoplanet-Detection\models\ml\scaler.pkl")
print("Scaler saved successfully!")

Scaler saved successfully!
